In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
df = pd.read_csv("FA-KES-Dataset.csv", encoding = "latin1")
true = pd.read_csv("True.csv")
fake= pd.read_csv("Fake.csv")

In [ ]:
df

In [ ]:
true['labels'] = 1
fake['labels'] = 0

In [ ]:
true.head()

In [ ]:
fake.head()

In [ ]:
news = pd.concat([fake, true], axis=0)
news

In [ ]:
df.info()

In [ ]:
news.info()

In [ ]:
news = news.drop(['title','subject','date'],axis =1)
news

In [ ]:
news.rename(columns={
    'text':'article_content'
},inplace='True')

In [ ]:
news

In [ ]:
df.isnull().sum()

In [ ]:
df=df.drop(['unit_id','article_title','source','date','location'],axis=1)

In [ ]:
df = pd.concat([df, news],axis = 0)

In [ ]:
df

In [ ]:
df=df.sample(frac=1)

In [ ]:
df.head()

In [ ]:
df.reset_index(inplace=True)
df=df.drop(['index'], axis =1)
df.head()

In [ ]:
def wordopt(text):
    #convert to lowercase
    text=text.lower()

    #remove URLS
    text = re.sub(r'https?://\S+|www\.\S+','', text)

    #remove HTML tags
    text = re.sub(r'<.*?>','',text)

    #remove punctuation
    text = re.sub(r'[^\w\s]','',text)

    #remove digits
    text = re.sub(r'\d','',text)

    #replace newline characters with space
    text = re.sub(r'\n',' ',text)

    return text

In [ ]:
df['article_content'] = df['article_content'].apply(wordopt)

In [ ]:
df['article_content']

In [ ]:
x = df['article_content']
y = df['labels']

In [ ]:
x

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorization = TfidfVectorizer()

In [ ]:
xv_train = vectorization.fit_transform(x_train)

In [ ]:
xv_test = vectorization.transform(x_test)

In [ ]:
xv_train
xv_test

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
LR = LogisticRegression()

In [ ]:
LR.fit(xv_train, y_train)

In [ ]:
pred_lr = LR.predict(xv_test)

In [ ]:
LR.score(xv_test, y_test)

In [ ]:
print(classification_report(y_test, pred_lr))

In [ ]:
def output_label(n):
    if n == 0:
        return "Fake News"
    elif n==1:
        return "True News"

In [ ]:
def manual_testing(news):
    testing_news = {"text": [news]}  # Corrected syntax for defining dictionary
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)  # Assuming 'vectorization' is your vectorizer object

    pred_lr = LR.predict(new_xv_test)
    # pred_dtc = dtc.predict(new_xv_test)
    # pred_gbc = gbc.predict(new_xv_test)
    # pred_rfc = rfc.predict(new_xv_test)

    return "LR Prediction: {}".format(
        output_label(pred_lr[0])
        # output_label(pred_gbc[0]),
        # output_label(pred_rfc[0])
    )

In [ ]:
news_article = str(input())

In [ ]:
manual_testing(news_article)